# FoodMacros - Nutrition5k RGB Mass Regression (Google Colab)

This is the single executable notebook for the **primary weight-estimation approach**:
RGB image -> MobileNetV2 -> food mass in grams, trained on Nutrition5k.

It uses the existing `bhumi` branch source code and does **not** retrain the classifier or YOLO detector.

## Before running

1. Zip your local Nutrition5k folder (it must contain `metadata/` and `imagery/`).
2. Upload the zip to Google Drive as `MyDrive/nutrition5k.zip`.
3. Select **Runtime -> Change runtime type -> GPU (T4)**.
4. Run the cells in order, top to bottom.

The dataset in this project has only 3,260 valid image + mass pairs because 1,506 metadata dishes
have no matching image and 240 rows have zero/invalid mass. All metrics produced here describe
that verified subset; no metric values are pre-filled or invented.

The final optional cell demonstrates classifier + nutrition output. It is skipped automatically
if the classifier artifact or Git LFS files are unavailable.


In [ ]:
import platform
import shutil
import subprocess
import sys
import tensorflow as tf

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)

print("Python:", platform.python_version())
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/FoodMacros")
REPO_URL = "https://github.com/Biswasrita/FoodMacros.git"
BRANCH = "bhumi"

if not (REPO_DIR / "src" / "foodmacros").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    print("Repository already present:", REPO_DIR)

os.chdir(REPO_DIR)
subprocess.run(["apt-get", "-qq", "install", "-y", "git-lfs"], check=False)
subprocess.run(["git", "lfs", "install"], check=False)
lfs = subprocess.run(["git", "lfs", "pull"], check=False)
if lfs.returncode != 0:
    print("LFS pull failed; optional classification/nutrition cell will be skipped.")


In [ ]:
import importlib
import subprocess
import sys

sys.path.insert(0, str(REPO_DIR / "src"))
print("Source path:", REPO_DIR / "src")

for name in ["tensorflow", "numpy", "pandas", "sklearn", "PIL", "matplotlib"]:
    try:
        module = importlib.import_module(name)
        print(f"{name}: {getattr(module, '__version__', 'OK')}")
    except Exception as exc:
        print(f"{name}: MISSING ({exc})")

try:
    import sklearn  # noqa: F401
except Exception:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "scikit-learn"],
        check=True,
    )

import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))


In [ ]:
import zipfile
from pathlib import Path

LOCAL_ROOT = Path("/content/nutrition5k")
DRIVE_ROOT = Path("/content/drive/MyDrive")


def looks_like_nutrition5k(path: Path) -> bool:
    return (
        path.is_dir()
        and any(path.glob("metadata/dish_metadata*.csv"))
        and (path / "imagery").is_dir()
    )


direct_candidates = [
    LOCAL_ROOT,
    DRIVE_ROOT / "nutrition5k",
    DRIVE_ROOT / "nutrition5k_dataset",
    Path("/content/nutrition5k_dataset"),
]
NUTRITION5K_ROOT = next(
    (path for path in direct_candidates if looks_like_nutrition5k(path)), None
)

if NUTRITION5K_ROOT is None:
    zip_candidates = [
        DRIVE_ROOT / "nutrition5k.zip",
        DRIVE_ROOT / "nutrition5k_dataset.zip",
    ]
    zip_candidates += sorted(DRIVE_ROOT.glob("*nutrition5k*.zip"))
    zip_path = next((p for p in zip_candidates if p.exists()), None)

    if zip_path is not None:
        print("Extracting:", zip_path)
        with zipfile.ZipFile(zip_path) as archive:
            archive.extractall("/content")
    else:
        print("No nutrition5k zip found in Drive; checking existing Colab folders.")

    for path in direct_candidates:
        if looks_like_nutrition5k(path):
            NUTRITION5K_ROOT = path
            break

if NUTRITION5K_ROOT is None:
    hits = list(Path("/content").rglob("dish_metadata_cafe1.csv"))
    if hits:
        NUTRITION5K_ROOT = hits[0].parent.parent

if NUTRITION5K_ROOT is None or not looks_like_nutrition5k(NUTRITION5K_ROOT):
    raise FileNotFoundError(
        "Nutrition5k not found. Upload nutrition5k.zip to MyDrive/nutrition5k.zip "
        "with metadata/ and imagery/ at the archive root."
    )

print("NUTRITION5K_ROOT =", NUTRITION5K_ROOT)


In [ ]:
from foodmacros.weight.mass_dataset import build_weight_dataset

ARTIFACTS = Path("/content/FoodMacros/artifacts/weight_dataset")
df, stats = build_weight_dataset(
    str(NUTRITION5K_ROOT),
    output_csv=ARTIFACTS / "all_pairs.csv",
)

print("Nutrition5k weight dataset")
print("--------------------------")
for key, value in stats.items():
    print(f"{key}: {value}")

assert len(df) > 0, "No valid image + mass pairs were found."


In [ ]:
from foodmacros.weight.mass_dataset import split_by_dish_id

train_df, val_df, test_df = split_by_dish_id(df, 0.7, 0.15, 0.15, seed=42)

for name, part in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print(
        f"{name}: dishes={part['dish_id'].nunique()}, images={len(part)}, "
        f"mass mean={part['mass_g'].mean():.2f} g"
    )


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

sample = train_df.sample(min(9, len(train_df)), random_state=42)
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for axis, (_, row) in zip(axes.ravel(), sample.iterrows()):
    axis.imshow(Image.open(row["image_path"]).convert("RGB"))
    axis.set_title(f"{row['dish_id']}\n{row['mass_g']:.1f} g")
    axis.axis("off")
for axis in axes.ravel()[len(sample):]:
    axis.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
for name, part in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print(f"\n{name}: n={len(part)}, dishes={part['dish_id'].nunique()}")
    print(part["mass_g"].describe())


In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    "scripts/train_weight.py",
    "--nutrition5k", str(NUTRITION5K_ROOT),
    "--output", "/content/FoodMacros/models/weight",
    "--artifacts", "/content/FoodMacros/artifacts/weight_dataset",
    "--eval-plots", "/content/FoodMacros/artifacts/weight_evaluation",
    "--epochs", "40",
    "--batch-size", "16",
    "--img-size", "224",
    "--fine-tune",
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True, cwd="/content/FoodMacros")


In [ ]:
from IPython.display import Image, display

EVAL_DIR = Path("/content/FoodMacros/artifacts/weight_evaluation")
curve_path = EVAL_DIR / "training_curves.png"
if curve_path.exists():
    display(Image(filename=str(curve_path)))
else:
    print("Missing:", curve_path)


In [ ]:
for name in [
    "actual_vs_predicted.png",
    "error_distribution.png",
    "mae_by_range.png",
]:
    path = EVAL_DIR / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))
    else:
        print("Missing:", path)


In [ ]:
import json

metrics_path = Path("/content/FoodMacros/models/weight/test_metrics.json")
if metrics_path.exists():
    print(json.dumps(json.loads(metrics_path.read_text()), indent=2))
else:
    print("Missing:", metrics_path)


In [ ]:
import json

import pandas as pd
import tensorflow as tf
from PIL import Image

from foodmacros.data.preprocessing import load_image
from foodmacros.weight.regression import (
    get_regression_preprocess_input,
    predict_mass_only_from_array,
)

MODEL_DIR = Path("/content/FoodMacros/models/weight")
norm = json.loads((MODEL_DIR / "normalization.json").read_text())
model = tf.keras.models.load_model(MODEL_DIR / "mass_regressor.keras", compile=False)
preprocess_fn = get_regression_preprocess_input(norm.get("backbone", "mobilenetv2"))
img_size = int(norm.get("img_size", 224))

rows = []
for _, row in test_df.sample(min(9, len(test_df)), random_state=7).iterrows():
    image = load_image(row["image_path"], img_size)
    predicted = predict_mass_only_from_array(
        model, preprocess_fn, image, img_size, norm["mean"], norm["std"]
    )
    rows.append(
        {
            "image_path": row["image_path"],
            "dish_id": row["dish_id"],
            "actual_g": float(row["mass_g"]),
            "predicted_g": float(predicted),
            "abs_error_g": abs(float(predicted) - float(row["mass_g"])),
        }
    )

preview = pd.DataFrame(rows)
display(preview)

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for axis, (_, row) in zip(axes.ravel(), preview.iterrows()):
    axis.imshow(Image.open(row["image_path"]).convert("RGB"))
    axis.set_title(f"actual {row['actual_g']:.0f} g\npred {row['predicted_g']:.0f} g")
    axis.axis("off")
for axis in axes.ravel()[len(preview):]:
    axis.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
import subprocess
import sys

sample_path = test_df.iloc[0]["image_path"]
subprocess.run(
    [
        sys.executable,
        "scripts/predict_weight.py",
        "--image", str(sample_path),
        "--model", "/content/FoodMacros/models/weight",
    ],
    check=True,
    cwd="/content/FoodMacros",
)


In [ ]:
import subprocess
import sys
from pathlib import Path

classifier_path = Path("/content/FoodMacros/models/classifier/classifier.keras")
if classifier_path.exists() and classifier_path.stat().st_size > 1_000_000:
    subprocess.run(
        [
            sys.executable,
            "scripts/run_analysis.py",
            str(sample_path),
            "--regressor-dir", "/content/FoodMacros/models/weight",
        ],
        check=False,
        cwd="/content/FoodMacros",
    )
else:
    print(
        "Optional end-to-end demo skipped: classifier artifact is missing or is a Git LFS pointer."
    )


In [ ]:
import shutil

DEST = Path("/content/drive/MyDrive/FoodMacros_weight")
DEST.mkdir(parents=True, exist_ok=True)

copy_map = {
    Path("/content/FoodMacros/models/weight"): DEST / "models_weight",
    Path("/content/FoodMacros/artifacts/weight_evaluation"): DEST / "weight_evaluation",
    Path("/content/FoodMacros/artifacts/weight_dataset"): DEST / "weight_dataset",
}
for source, destination in copy_map.items():
    if source.exists():
        shutil.copytree(source, destination, dirs_exist_ok=True)
        print("Copied:", source, "->", destination)
    else:
        print("Missing (skipped):", source)

print("Saved results to:", DEST)


## Outputs to collect for your submission

After the run, download these from `MyDrive/FoodMacros_weight/`:

- `models_weight/mass_regressor.keras` - trained RGB weight model
- `models_weight/normalization.json` - mass normalization statistics
- `models_weight/test_metrics.json` - actual test MAE / RMSE / R2 / MAPE and baseline
- `weight_evaluation/training_curves.png` - training and validation loss/MAE
- `weight_evaluation/actual_vs_predicted.png`
- `weight_evaluation/error_distribution.png`
- `weight_evaluation/mae_by_range.png`
- `weight_dataset/train.csv`, `validation.csv`, `test.csv`, `all_pairs.csv`

The prediction cell prints a table of actual vs predicted grams for nine test dishes, and the CLI
cell prints `Estimated food weight: ... g` for one test image. These are actual model outputs for
the verified 3,260-pair subset, not pre-filled results.
